
ALL Imports and loading of data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # For plotting

In [4]:
# change the entire code to function instead of snippets
df = pd.read_csv('/content/clean_dataset.csv')

# Shuffle the indices of the DataFrame
shuffled_indices = np.random.permutation(df.index)

# Calculate the sizes of training, validation, and test sets
total_size = len(df)
train_size = int(total_size * 0.5)
val_size = int(total_size * 0.4)
test_size = total_size - train_size - val_size

# Split the shuffled indices into training, validation, and test sets
train_indices = shuffled_indices[:train_size]
val_indices = shuffled_indices[train_size:train_size+val_size]
test_indices = shuffled_indices[train_size+val_size:]

# Create training, validation, and test DataFrames using the selected indices
train_data = df.loc[train_indices]
val_data = df.loc[val_indices]
test_data = df.loc[test_indices]

dubai_q = df[df['Label'] == 'Dubai']['Q10'].tolist()
paris_q = df[df['Label'] == 'Paris']['Q10'].tolist()
new_york_q = df[df['Label'] == 'New York City']['Q10'].tolist()
rio_q = df[df['Label'] == 'Rio de Janeiro']['Q10'].tolist()

LOOKING INTO HOW DATA LOOKS LIKE

In [5]:
# train_data.describe()

In [6]:
# plt.title("Box Plot Showing the Distribution of LIMIT_BAL")
# plt.boxplot(train_data["Q1"])

In [7]:
columns_to_process = ["Q1", "Q2", "Q3", "Q4"]

# Calculate the mean for each column, excluding invalid entries
means = {}
for column in columns_to_process:
    valid_values = train_data[column][(train_data[column] >= 1) & (train_data[column] <= 5)]
    means[column] = valid_values.mean()

# Function to replace values based on the condition
def replace_values(row, column_name):
    if pd.isna(row) or row < 1 or row > 5:
        return means[column_name]
    else:
        return row

# Apply the function to each column in the list
for column in columns_to_process:
    train_data[column] = train_data[column].apply(replace_values, column_name=column)

means = {}
for column in columns_to_process:
    valid_values = val_data[column][(val_data[column] >= 1) & (val_data[column] <= 5)]
    means[column] = valid_values.mean()

# Function to replace values based on the condition
def replace_values(row, column_name):
    if pd.isna(row) or row < 1 or row > 5:
        return means[column_name]
    else:
        return row

# Apply the function to each column in the list
for column in columns_to_process:
    val_data[column] = val_data[column].apply(replace_values, column_name=column)

means = {}
for column in columns_to_process:
    valid_values = test_data[column][(test_data[column] >= 1) & (test_data[column] <= 5)]
    means[column] = valid_values.mean()

# Function to replace values based on the condition
def replace_values(row, column_name):
    if pd.isna(row) or row < 1 or row > 5:
        return means[column_name]
    else:
        return row

# Apply the function to each column in the list
for column in columns_to_process:
    test_data[column] = test_data[column].apply(replace_values, column_name=column)


In [8]:
categories_q5 = ["Friends", "Co-worker", "Partner", "Sibling"]
for category in categories_q5:
    train_data[category] = train_data['Q5'].apply(lambda x: 1 if category in str(x) else 0)

categories_q5 = ["Friends", "Co-worker", "Partner", "Sibling"]
for category in categories_q5:
    val_data[category] = val_data['Q5'].apply(lambda x: 1 if category in str(x) else 0)

categories_q5 = ["Friends", "Co-worker", "Partner", "Sibling"]
for category in categories_q5:
    test_data[category] = test_data['Q5'].apply(lambda x: 1 if category in str(x) else 0)


In [9]:
def extract_value(row, category):
    items = str(row).split(',')
    category_map = {item.split('=>')[0]: (float(item.split('=>')[1]) if item.split('=>')[1] else np.nan)
                    for item in items if '=>' in item}
    return category_map.get(category, np.nan)

categories_q6 = ["Skyscrapers", "Art and Music", "Carnival", "Cuisine", "Sport", "Economic"]

# Apply the extract_value function to each category
for category in categories_q6:
    train_data[category] = train_data['Q6'].apply(lambda x: extract_value(x, category))

# Calculate the nanmean for each of the new columns
means_q6 = {category: np.nanmean(train_data[category]) for category in categories_q6}

# Replace NaN values in each category column with the column's mean
for category in categories_q6:
    train_data[category].fillna(means_q6[category], inplace=True)

# Apply the extract_value function to each category
for category in categories_q6:
    val_data[category] = val_data['Q6'].apply(lambda x: extract_value(x, category))

# Calculate the nanmean for each of the new columns
means_q6 = {category: np.nanmean(val_data[category]) for category in categories_q6}

# Replace NaN values in each category column with the column's mean
for category in categories_q6:
    val_data[category].fillna(means_q6[category], inplace=True)

# Apply the extract_value function to each category
for category in categories_q6:
    test_data[category] = test_data['Q6'].apply(lambda x: extract_value(x, category))

# Calculate the nanmean for each of the new columns
means_q6 = {category: np.nanmean(test_data[category]) for category in categories_q6}

# Replace NaN values in each category column with the column's mean
for category in categories_q6:
    test_data[category].fillna(means_q6[category], inplace=True)

In [10]:
# Drop the original 'Q5' and 'Q6' columns
train_data = train_data.drop(['Q5', 'Q6'], axis=1)

val_data = val_data.drop(['Q5', 'Q6'], axis=1)

test_data = test_data.drop(['Q5', 'Q6'], axis=1)

In [11]:
placeholder_mean = np.nanmean(pd.to_numeric(train_data['Q7'].replace(',', ''), errors='coerce'))
def adjust_q7(value):
    if pd.isna(value):
        return placeholder_mean
    else:
        # Remove commas before conversion to int
        value_no_commas = str(value).replace(',', '')
        return min(max(float(value_no_commas), -20), 70)

train_data['Q7'] = train_data['Q7'].apply(adjust_q7)

placeholder_mean = np.nanmean(pd.to_numeric(val_data['Q7'].replace(',', ''), errors='coerce'))

val_data['Q7'] = val_data['Q7'].apply(adjust_q7)

placeholder_mean = np.nanmean(pd.to_numeric(test_data['Q7'].replace(',', ''), errors='coerce'))

test_data['Q7'] = test_data['Q7'].apply(adjust_q7)

In [12]:
placeholder_mean = np.nanmean(pd.to_numeric(train_data['Q8'].replace(',', ''), errors='coerce'))
def adjust_q8(value):
    if pd.isna(value):
        return placeholder_mean
    else:
        # Remove commas before conversion to int
        value_no_commas = str(value).replace(',', '')
        return min(max(float(value_no_commas), 1), 20)

train_data['Q8'] = train_data['Q8'].apply(adjust_q8)

placeholder_mean = np.nanmean(pd.to_numeric(val_data['Q8'].replace(',', ''), errors='coerce'))

val_data['Q8'] = val_data['Q8'].apply(adjust_q8)

placeholder_mean = np.nanmean(pd.to_numeric(test_data['Q8'].replace(',', ''), errors='coerce'))

test_data['Q8'] = test_data['Q8'].apply(adjust_q8)


In [13]:
placeholder_mean = np.nanmean(pd.to_numeric(train_data['Q9'].replace(',', ''), errors='coerce'))

def adjust_q9(value):
    if pd.isna(value):
        return placeholder_mean
    else:
        # Remove commas before conversion to int
        value= str(value).replace(',', '')
        return min(max(float(value), 1), 50)

train_data['Q9'] = train_data['Q9'].apply(adjust_q9)

placeholder_mean = np.nanmean(pd.to_numeric(val_data['Q9'].replace(',', ''), errors='coerce'))

val_data['Q9'] = val_data['Q9'].apply(adjust_q9)

placeholder_mean = np.nanmean(pd.to_numeric(test_data['Q9'].replace(',', ''), errors='coerce'))

test_data['Q9'] = test_data['Q9'].apply(adjust_q9)

In [14]:
import string

# Basic text preprocessing function
def preprocess_text(text):
    if type(text) == float:
        text = ""
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize the text
    tokens = text.split()
    return tokens

# Custom list of English stopwords
custom_stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves',
                    'you', 'your', 'yours', 'yourself',
                    'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her',
                    'hers', 'herself', 'it', 'its', 'itself',
                    'they', 'them', 'their', 'theirs', 'themselves', 'what',
                    'which', 'who', 'whom', 'this', 'that',
                    'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be',
                    'been', 'being', 'have', 'has', 'had',
                    'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the',
                    'and', 'but', 'if', 'or', 'because',
                    'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with',
                    'about', 'against', 'between', 'into',
                    'through', 'during', 'before', 'after', 'above', 'below',
                    'to', 'from', 'up', 'down', 'in', 'out',
                    'on', 'off', 'over', 'under', 'again', 'further', 'then',
                    'once', 'here', 'there', 'when', 'where',
                    'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
                    'most', 'other', 'some', 'such', 'no',
                    'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too',
                    'very', 's', 't', 'can', 'will', 'just',
                    'don', 'should', 'now'] + ['thus', 'foie', 'rather',
                                               'takes', 'find', 'main',
                                               'saying', 'spreadin', 'stub',
                                               'hon', 'annual', 'guess', 'est',
                                               'fk', 'mundo', 'says', 'size',
                                               'third', 'gain', 'air',
                                               'perfect', 'help', 'end', 'getâ',
                                               'low', 'pack', 'pot', 'grand',
                                               'riding', 'turned', 'uhâ',
                                               'workâ', 'short', 'take', 'lazy',
                                               'anywhere', 'never', 'places',
                                               'second', 'dog', 'ba', 'else',
                                               'cristo', 'enough', 'mans', 'vi',
                                               'long', 'problems', 'points',
                                               'together', 'charles', 'sleep',
                                               'og', 'bright', 'hands',
                                               'toronto', 'loca', 'exists',
                                               'work', 'west', 'times', 'makes',
                                               'land', 'one', 'balanced',
                                               'haven', 'lisa', 'differences',
                                               'probably', 'industry',
                                               'enthusiasm', 'much', 'taxes',
                                               'trading', 'pretty', 'without',
                                               'countries', 'almost', 'awayâ',
                                               'birth', 'fall', 'moments',
                                               'big', 'build', 'ere', 'wants',
                                               'satisfy', 'gorgeous',
                                               'absolutely', 'paradise',
                                               'airlines', 'itemsâ', 'higher',
                                               'carrie', 'realize', 'later',
                                               'hub', 'time', 'absolute',
                                               'riots', 'disillusionment',
                                               'teenage', 'every', 'sing',
                                               'comfortable', 'prowess', 'sera',
                                               'willing', 'hotspot', 'early',
                                               'three', 'faster', 'summers',
                                               'another', 'accept', 'staying',
                                               'hello', 'happiness', 'taylor',
                                               'gods', 'pursuit', 'endless',
                                               'copious', 'rainy', 'technology',
                                               'lives', 'feeling', 'could',
                                               'cool', 'hope', 'sandy',
                                               'thereafter', 'pissed', 'sound',
                                               'wet', 'extremely', 'give',
                                               'turn', 'gang', 'works', 'see',
                                               'wilde', 'relentless', 'high',
                                               'opportunities', 'better',
                                               'novo', 'tyson', 'leavin',
                                               'started', 'waiting',
                                               'extravagant', 'variety', 'died',
                                               'found', 'believe', 'dude',
                                               'ocean', 'oscar', 'parent',
                                               'bigger', 'loving', 'ought',
                                               'week', 'birds', 'way',
                                               'tomorrow', 'wait', 'lot',
                                               'deserves', 'tall', 'master',
                                               'hereâ', 'enduring', 'authentic',
                                               'views', 'whatâ', 'man',
                                               'please', 'geographical', 'dont',
                                               'preservences', 'fullest',
                                               'whispers', 'hudson',
                                               'watercolor', 'specific',
                                               'thats', 'challenges', 'purpose',
                                               'worldâ', 'quote', 'testament',
                                               'element', 'strength', 'back',
                                               'modernday', 'close',
                                               'overwatch', 'gras', 'heist',
                                               'stars', 'stone', 'kent',
                                               'allons', 'whitewashed', 'book',
                                               'donc', 'criticism', 'gather',
                                               'alone', 'looking', 'enter',
                                               'play', 'finish', 'steven',
                                               'zooyork', 'vida', 'mean',
                                               'nocturnal', 'kid', 'family',
                                               'partying', 'chaois', 'ninja',
                                               'brings', 'written', 'boundless',
                                               'plait', 'deal', 'ammazing',
                                               'overly', 'torch', 'empty',
                                               'carefree', 'playing', 'masked',
                                               'great', 'somehow', 'carry',
                                               'rome', 'anyone', 'newww',
                                               'hollow', 'famous', 'right',
                                               'sky', 'talk', 'okay',
                                               'constantly', 'next', 'raised',
                                               'happen', 'london', 'walk',
                                               'colourful', 'daring',
                                               'progress', 'requires', 'moment',
                                               'century', 'world', 'teaches',
                                               'women', 'blessed', 'nothin',
                                               'either', 'fossil', 'term',
                                               'thereâ', 'campeo', 'sacrifice',
                                               'peaceful', 'pick', 'appleâ',
                                               'two', 'sands', 'cozy', 'memory',
                                               'prosperityâ', 'direction',
                                               'different', 'garbage', 'also',
                                               'blue', 'nice', 'held',
                                               'feijoada', 'surface',
                                               'carnival', 'en', 'si',
                                               'spectacular', 'universes',
                                               'reality', 'dance', 'lose',
                                               'live', 'active', 'parody',
                                               'mafia', 'la', 'ed', 'mona',
                                               'obvious', 'canâ', 'becoming',
                                               'barefoot', 'like', 'outdated',
                                               'drop', 'crunches', 'property',
                                               'uh', 'workers', 'loaf',
                                               'powers', 'reach', 'cold',
                                               'shall', 'taking', 'year',
                                               'someone', 'ohh', 'antiquated',
                                               'mixed', 'job', 'old', 'players',
                                               'wanted', 'named', 'romanticâ',
                                               'upon', 'part', 'goes', 'crust',
                                               'forest', 'power', 'change',
                                               'symphony', 'enthusiastic',
                                               'show', 'joseph', 'emptiness',
                                               'feel', 'joga', 'natural',
                                               'cannot', 'stepping', 'wings',
                                               'pense', 'fly', 'smell',
                                               'alright', 'color', 'cream',
                                               'fancy', 'gigantic', 'ever',
                                               'care', 'fun', 'mere', 'statueâ',
                                               'lines', 'brodsky', 'colors',
                                               'imagined', 'apartment', 'cestâ',
                                               'race', 'jordan', 'moveable',
                                               'magnificent', 'thrives',
                                               'silence', 'economic', 'michael',
                                               'deep', 'mutant', 'six', 'firm',
                                               'judge', 'backed', 'continuous',
                                               'worm', 'extreme', 'earth',
                                               'rest', 'doctor', 'piece', 'fyi',
                                               'go', 'going', 'bob', 'martial',
                                               'enjoyed', 'turtles', 'ugliness',
                                               'err', 'best', 'roots', 'taken',
                                               'vibrant', 'walking', 'soul',
                                               'oasis', 'may', 'games',
                                               'stealing', 'exciting', 'samba',
                                               'streets', 'isnt', 'calculated',
                                               'goodbye', 'song', 'ahha',
                                               'side', 'spirit', 'reason',
                                               'soy', 'serenity', 'location',
                                               'flow', 'fansâ', 'rigorous',
                                               'wakin', 'question', 'tech',
                                               'elfâ', 'dickensâ', 'helen',
                                               'saves', 'health', 'feast',
                                               'aha', 'ft', 'share', 'vive',
                                               'still', 'pain', 'meet',
                                               'greatest', 'vests', 'keller',
                                               'burden', 'rose', 'nothing',
                                               'meaningful', 'dying', 'apart',
                                               'heights', 'inclusive', 'away',
                                               'naughty', 'donde', 'within',
                                               'underground', 'afford',
                                               'average', 'captured', 'brand',
                                               'train', 'trae', 'ends',
                                               'sculpture', 'beautiful',
                                               'politics', 'irving', 'chef',
                                               'jungle', 'insist', 'heads',
                                               'weak', 'replies', 'count',
                                               'general', 'seems',
                                               'grandmaster', 'start', 'starts',
                                               'stinson', 'furious',
                                               'reflected', 'prosperity',
                                               'vivid', 'itget', 'answer',
                                               'tasting', 'las', 'slums',
                                               'heaven', 'middle', 'know',
                                               'insatiate', 'rumble',
                                               'inspires', 'harder', 'yes',
                                               'tennis', 'dj', 'street',
                                               'making', 'pizza', 'fraternity',
                                               'exchangeâ', 'say', 'center',
                                               'rotten', 'hearts', 'bad',
                                               'pockets', 'centre', 'hell',
                                               'thought', 'jealous', 'pen',
                                               'universe', 'cultures', 'cook',
                                               'unconditionally', 'decisions',
                                               'head', 'league', 'wisdom',
                                               'landmarks', 'east', 'eat',
                                               'kelk', 'tears', 'tell',
                                               'business', 'ow', 'yet',
                                               'income', 'goddamn', 'fight',
                                               'course', 'disgusting', 'tomato',
                                               'mark', 'influential', 'carried',
                                               'touched', 'jose', 'exist',
                                               'thanks', 'divide', 'ball',
                                               'united', 'bandolero',
                                               'multicultural', 'belongs',
                                               'climate', 'redemption',
                                               'science', 'anxious', 'night',
                                               'patrick', 'member', 'code',
                                               'drinks', 'digennaro', 'toto',
                                               'oliveira', 'unlike', 'entire',
                                               'comes', 'revolution', 'story',
                                               'license', 'waves',
                                               'experienced', 'ive',
                                               'imposssible', 'modernityâ',
                                               'many', 'oh', 'liable',
                                               'present', 'ice', 'wildest',
                                               'miraculous', 'welcome',
                                               'secrets', 'alicia', 'waitressâ',
                                               'jobs', 'lived', 'diversity',
                                               'useless', 'days', 'sil',
                                               'unity', 'x', 'suit', 'wins',
                                               'grew', 'meant', 'instantly',
                                               'exploit', 'redeemer', 'bolder',
                                               'hit', 'smart', 'walt',
                                               'particular', 'immigrants',
                                               'overrated', 'destination',
                                               'safety', 'specifically',
                                               'richâ', 'fries', 'safe', 'clan',
                                               'dorothy', 'glitz', 'houses',
                                               'elderado', 'hard', 'perhaps',
                                               'price', 'rules', 'broke',
                                               'lots', 'writes', 'simple',
                                               'hey', 'seasons', 'letter',
                                               'corrupts', 'musical', 'clears',
                                               'father', 'filled', 'noisy',
                                               'gaston', 'dessert',
                                               'containing', 'possibly',
                                               'crime', 'summer', 'forget',
                                               'hamilton', 'inequality', 'must',
                                               'designed', 'gets', 'symbol',
                                               'impossible', 'record', 'swift',
                                               'jr', 'asked', 'forests',
                                               'cards', 'adventure', 'chosen',
                                               'nature', 'nowhereâ', 'first',
                                               'sunkissed', 'whimsicalâ',
                                               'lets', 'even', 'sleepless',
                                               'possible', 'existence', 'day',
                                               'lãºcio', 'elegance', 'david',
                                               'moneyâ', 'patagonia',
                                               'enjoying', 'equality', 'bigâ',
                                               'succeed', 'flyâ', 'human',
                                               'anything', 'nights', 'stomach',
                                               'cities', 'real', 'classmates',
                                               'destiny', 'today', 'somewhat',
                                               'thoreau', 'twice', 'dream',
                                               'keep', 'flashy', 'decorations',
                                               'failure', 'however',
                                               'attractionâ', 'exceeded',
                                               'despair', 'youd', 'pony',
                                               'dreams', 'haute', 'connect',
                                               'city', 'five', 'put', 'learn',
                                               'classic', 'whole',
                                               'infrastructure', 'us', 'feels',
                                               'economically', 'concrete',
                                               'confucius', 'bugs', 'people',
                                               'news', 'system', 'experience',
                                               'thinking', 'settle', 'stole',
                                               'walkin', 'opportunity', 'light',
                                               'follower', 'violations',
                                               'vision', 'intersection', 'gas',
                                               'need', 'junk', 'huge', 'artist',
                                               'happened', 'crowded',
                                               'nightmare', 'playground',
                                               'house', 'worry', 'jules',
                                               'wolf', 'idk', 'ambitions',
                                               'sorry', 'argument', 'beyond',
                                               'havent', 'knowledgeable',
                                               'route', 'vacation',
                                               'complicatedâ', 'despite',
                                               'freedom', 'state', 'might',
                                               'cant', 'yeah', 'heartâ',
                                               'emily', 'ambition', 'magic',
                                               'sku', 'flash', 'sell',
                                               'seasideâ', 'hierarchy',
                                               'wheres', 'angry', 'inspire',
                                               'downtown', 'gonna', 'dine',
                                               'sao', 'treasure', 'shining',
                                               'leader', 'die', 'rhythms', 'â',
                                               'quotes', 'memories', 'quit',
                                               'crazy', 'landscape', 'cover',
                                               'parrots', 'redentor',
                                               'prosperous', 'culturally',
                                               'millions', 'luck', 'excessâ',
                                               'fearless', 'fused', 'living',
                                               'steve', 'mouth', 'fact', 'toâ',
                                               'dries', 'line', 'silver',
                                               'understanding', 'aunque', 'shy',
                                               'discover', 'years', 'black',
                                               'nations', 'stupid', 'amounts',
                                               'professional', 'brothas', 'ofâ',
                                               'sunny', 'happyâ', 'cultural',
                                               'wonder', 'blood', 'wearing',
                                               'possibilities', 'eyes', 'homes',
                                               'future', 'stays', 'exchange',
                                               'presenting', 'bought', 'ones',
                                               'ill', 'lexicon', 'girl',
                                               'facade', 'rural', 'made',
                                               'controls', 'laidback',
                                               'popular', 'horses', 'full',
                                               'rhythm', 'viva', 'place',
                                               'yellow', 'yorkðÿžµ', 'men',
                                               'eurocentric', 'janeiroâ', 'pay',
                                               'ddrop', 'ho', 'e', 'word',
                                               'joy', 'act', 'situations',
                                               'mood', 'aldo', 'ride', 'club',
                                               'specter', 'wannabe', 'senses',
                                               'choice', 'youre', 'imagine',
                                               'succumb', 'ur', 'de', 'souls',
                                               'game', 'baby', 'keys', 'large',
                                               'son', 'architecture', 'theâ',
                                               'ashamed', 'raise', 'former',
                                               'worrying', 'death', 'gasoline',
                                               'aux', 'projects', 'everything',
                                               'el', 'painting', 'seen', 'un',
                                               'peter', 'lost', 'coldâ', 'cost',
                                               'favelas', 'romans',
                                               'similarities', 'lincoln',
                                               'white', 'weather', 'ostriches',
                                               'since', 'digan', 'source',
                                               'awesome', 'trips', 'less',
                                               'life', 'koch', 'annie',
                                               'affordable', 'keeps', 'sunâ',
                                               'though', 'guests', 'hot',
                                               'social', 'lindsey', 'artistic',
                                               'coolest', 'beauty', 'loveâ',
                                               'hear', 'driving', 'alive',
                                               'abraham', 'plane', 'disney',
                                               'tower', 'festive', 'alot',
                                               'passionate', 'respectâ',
                                               'remember', 'person', 'hosted',
                                               'celebrate', 'wallet', 'flags',
                                               'gustave', 'setback', 'sings',
                                               'patisserie', 'think', 'cake',
                                               'plate', 'heightsâ', 'wherever',
                                               'erase', 'clouds', 'gathers',
                                               'acquired', 'would',
                                               'confidently', 'shape', 'idea',
                                               'loud', 'antidote', 'along',
                                               'built', 'dalai', 'everywhere',
                                               'amâ', 'bonita', 'always',
                                               'kylian', 'really', 'activities',
                                               'charo', 'happens', 'resides',
                                               'thing', 'want', 'things',
                                               'lucky', 'energy', 'richer',
                                               'getgegetgeget', 'tumbrels',
                                               'spread', 'timeless',
                                               'opprtunity', 'springing',
                                               'lasts', 'sins', 'community',
                                               'exercise', 'infinity', 'harsh',
                                               'buy', 'glitters', 'enjoy',
                                               'lively', 'poetry', 'barely',
                                               'cityscapeâ', 'face', 'blu',
                                               'eats', 'cantâ', 'worthwhile',
                                               'psycho', 'paintbrush', 'used',
                                               'serves', 'floating', 'young',
                                               'henry', 'artists', 'fear',
                                               'enjoys', 'lies', 'tells',
                                               'history', 'bailao', 'delicate',
                                               'version', 'trash', 'blank',
                                               'stayâ', 'tiny', 'meets',
                                               'definitely', 'lower', 'golden',
                                               'aller', 'played', 'theyll',
                                               'stops', 'viral', 'become',
                                               'wish', 'poverty', 'number',
                                               'party', 'eagle', 'vegas',
                                               'break', 'morning', 'theres',
                                               'jive', 'surrounded', 'fast',
                                               'wrapper', 'based', 'something',
                                               'soup', 'makeâ', 'excellent',
                                               'peopleâ', 'true', 'court',
                                               'puffy', 'mi', 'needs',
                                               'matuidi', 'sleeps', 'bailar',
                                               'poorer', 'bedbugs',
                                               'accountants', 'home', 'visit',
                                               'flying', 'style', 'diet',
                                               'melting', 'sounds', 'laugh',
                                               'homeâ', 'slice', 'rely', 'lama',
                                               'shaffer', 'celebration',
                                               'greed', 'cats', 'get',
                                               'stories', 'actually', 'means',
                                               'thousand', 'round', 'country',
                                               'come', 'felt', 'al', 'let',
                                               'scene', 'states', 'anymoreâ',
                                               'ohâ', 'fulfilment', 'panda',
                                               'selfdenial', 'sibling', 'case',
                                               'loss', 'lights', 'bed',
                                               'greatness', 'distinguishes',
                                               'roll', 'amazing', 'wall',
                                               'candy', 'worst', 'trap',
                                               'bullish', 'founded', 'well',
                                               'responsibility', 'make',
                                               'labour', 'score', 'gave',
                                               'strikes', 'look', 'proceeded',
                                               'classical', 'turns', 'mind',
                                               'illegal', 'appears',
                                               'naturalistic', 'super',
                                               'joanne', 'sun', 'ðÿžµwhen',
                                               'said', 'cheaper', 'um',
                                               'deserve', 'doesnt', 'jungles',
                                               'minutes', 'insideâ', 'good',
                                               'sunshine', 'esque', 'companies',
                                               'finds', 'jackson', 'bird',
                                               'luxury', 'landscapes', 'grow',
                                               'superpowers', 'swimming', 'hui',
                                               'worlds', 'restart',
                                               'understand', 'new', 'miracle',
                                               'devouring', 'maybe', 'marley',
                                               'janeiro', 'everyone', 'im',
                                               'builds', 'parties',
                                               'designating', 'little',
                                               'brokerage', 'realization',
                                               'bottom', 'area', 'feelings',
                                               'top', 'wrenching', 'bradshaw',
                                               'rent', 'economy', 'routine',
                                               'embarrass', 'name', 'harris',
                                               'none', 'wutang', 'crossroads',
                                               'succession', 'angery', 'happy',
                                               'probs', 'expectations', 'tbh',
                                               'youve', 'tunnels', 'roof',
                                               'coreta', 'use', 'already',
                                               'tradition', 'born']


# Function to create a vocabulary from a list of preprocessed sentences
def create_vocabulary(sentences):
    vocabulary = set()
    for sentence in sentences:
        words = preprocess_text(sentence)
        # Exclude custom stopwords from the vocabulary
        words = [word for word in words if
                 word not in custom_stopwords and word.isalpha() and not word.isdigit()]
        vocabulary.update(words)
    return list(vocabulary)

# Fill NaN values with empty strings
df['Q10'].fillna("", inplace=True)
train_data['Q10'].fillna("", inplace=True)
val_data['Q10'].fillna("", inplace=True)
test_data['Q10'].fillna("", inplace=True)

# Get all sentences from the training dataset
sentences = df['Q10'].tolist()

# Construct the global vocabulary using the training dataset
vocabulary = create_vocabulary(sentences)

# Function to create a vocabulary from a list of preprocessed sentences
# def create_vocabulary(sentences):
#     vocabulary = set()
#     for sentence in sentences:
#         words = preprocess_text(sentence)
#         # Exclude custom stopwords from the vocabulary
#         words = [word for word in words if
#                  word not in custom_stopwords and word.isalpha() and not word.isdigit()]
#         vocabulary.update(words)
#     return list(vocabulary)
#
# # Fill NaN values with empty strings
# df['Q10'].fillna("", inplace=True)
# train_data['Q10'].fillna("", inplace=True)
# val_data['Q10'].fillna("", inplace=True)
# test_data['Q10'].fillna("", inplace=True)
#
# # Get all sentences from the training dataset
# sentences = df['Q10'].tolist()
#
# # Construct the global vocabulary using the training dataset
# vocabulary = create_vocabulary(sentences)

# Calculate matching percentages for each label and update columns in train_data, val_data, and test_data
def calculate_matching_percentages(data, vocabulary, label, label_vector):
    matching_percentages = []
    for index in data.index:
        sentence = data.loc[index, 'Q10']
        bow_vector = bow_vectorize(sentence, label_vector)
        matching_percentages.append(np.mean(bow_vector))
    data[label] = matching_percentages
    return data


# Function to generate Bag of Words (BoW) vectors
def bow_vectorize(sentence, vocabulary):
    # Initialize vector with zeros
    vector = np.zeros(len(vocabulary))
    # Preprocess the sentence
    tokens = preprocess_text(sentence)
    # Count token occurrences
    for token in tokens:
        if token in vocabulary:
            vector[vocabulary.index(token)] += 1
    return vector

# Calculate BoW vectors for the entire dataset and update vocabulary columns
# def calculate_bow_vectors(data, vocabulary):
#     bow_matrix = []
#     for index in data.index:
#         sentence = data.loc[index, 'Q10']
#         bow_vector = bow_vectorize(sentence, vocabulary)
#         bow_matrix.append(bow_vector)
#     return bow_matrix

dubai_q = create_vocabulary(dubai_q)
paris_q = create_vocabulary(paris_q)
new_york_q = create_vocabulary(new_york_q)
rio_q = create_vocabulary(rio_q)

print(dubai_q)
print(paris_q)
print(new_york_q)
print(rio_q)

# Calculate matching percentages for each dataset and label
train_data = calculate_matching_percentages(train_data, vocabulary, 'Dubai', dubai_q)
train_data = calculate_matching_percentages(train_data, vocabulary, 'Paris', paris_q)
train_data = calculate_matching_percentages(train_data, vocabulary, 'New York City', new_york_q)
train_data = calculate_matching_percentages(train_data, vocabulary, 'Rio de Janeiro', rio_q)

val_data = calculate_matching_percentages(val_data, vocabulary, 'Dubai', dubai_q)
val_data = calculate_matching_percentages(val_data, vocabulary, 'Paris', paris_q)
val_data = calculate_matching_percentages(val_data, vocabulary, 'New York City', new_york_q)
val_data = calculate_matching_percentages(val_data, vocabulary, 'Rio de Janeiro', rio_q)

test_data = calculate_matching_percentages(test_data, vocabulary, 'Dubai', dubai_q)
test_data = calculate_matching_percentages(test_data, vocabulary, 'Paris', paris_q)
test_data = calculate_matching_percentages(test_data, vocabulary, 'New York City', new_york_q)
test_data = calculate_matching_percentages(test_data, vocabulary, 'Rio de Janeiro', rio_q)

# Drop the original 'Q10' column
train_data = train_data.drop(['Q10'], axis=1)
val_data = val_data.drop(['Q10'], axis=1)
test_data = test_data.drop(['Q10'], axis=1)

#
#
# # Calculate BoW vectors for each dataset
# train_bow = calculate_bow_vectors(train_data, vocabulary)
# val_bow = calculate_bow_vectors(val_data, vocabulary)
# test_bow = calculate_bow_vectors(test_data, vocabulary)
#
# # Convert bow_matrix to DataFrame
# train_bow_df = pd.DataFrame(train_bow, columns=vocabulary)
# val_bow_df = pd.DataFrame(val_bow, columns=vocabulary)
# test_bow_df = pd.DataFrame(test_bow, columns=vocabulary)
#
# # Reset indices to ensure alignment
# train_data.reset_index(drop=True, inplace=True)
# train_bow_df.reset_index(drop=True, inplace=True)
#
# # Reset indices to ensure alignment
# val_data.reset_index(drop=True, inplace=True)
# val_bow_df.reset_index(drop=True, inplace=True)
#
# # Reset indices to ensure alignment
# test_data.reset_index(drop=True, inplace=True)
# test_bow_df.reset_index(drop=True, inplace=True)
#
# # Concatenate BoW matrix to train_data, val_data, and test_data
# train_data = pd.concat([train_data, train_bow_df], axis=1)
# val_data = pd.concat([val_data, val_bow_df], axis=1)
# test_data = pd.concat([test_data, test_bow_df], axis=1)
#
# train_data = train_data.drop(['Q10'], axis=1)
#
# val_data = val_data.drop(['Q10'], axis=1)
#
# test_data = test_data.drop(['Q10'], axis=1)


['underbelly', 'burj', 'uae', 'tax', 'hypermodernized', 'love', 'skyscraperâ', 'slave', 'wealth', 'money', 'poor', 'class', 'skyscrapersâ', 'divided', 'ford', 'cristiano', 'skies', 'marvel', 'mian', 'finite', 'arabic', 'skyscraper', 'princes', 'sand', 'windy', 'soccer', 'dollars', 'nightlife', 'marvels', 'trade', 'tate', 'wealthy', 'lamborghini', 'sport', 'palm', 'fuel', 'khaled', 'himid', 'prince', 'limit', 'riches', 'architectural', 'arab', 'oil', 'influencers', 'heart', 'food', 'suite', 'yallah', 'artificial', 'dubaiâ', 'lavish', 'dubai', 'king', 'loved', 'passion', 'soulless', 'exploitation', 'rains', 'cars', 'clothing', 'tallest', 'innovation', 'petroleum', 'ronaldo', 'khalifa', 'desert', 'buildings', 'habibi', 'richest', 'islands', 'skyscrapers', 'skyline', 'emirates', 'honeymoon', 'expensive', 'modern', 'yalla', 'richness', 'opulence', 'beach', 'development', 'university', 'capitalism', 'gold', 'batman', 'luxurious', 'dubais', 'shallow', 'warm', 'excellence', 'futuristic', 'tour

In [15]:
train_data

,id,Q1,Q2,Q3,Q4,Q7,Q8,Q9,Label,Friends,...,Skyscrapers,Art and Music,Carnival,Cuisine,Sport,Economic,Dubai,Paris,New York City,Rio de Janeiro
290,499778,4.0,2.0,5.0,1.0,20.0,5.0,5.0,Dubai,1,...,6.0,3.0,1.0,2.0,4.0,5.0,0.018519,0.000000,0.022989,0.0
131,523461,5.0,3.0,4.0,3.0,25.0,4.0,3.0,Dubai,1,...,5.0,1.0,3.0,4.0,2.0,6.0,0.009259,0.000000,0.000000,0.0
778,602496,5.0,5.0,5.0,4.0,0.0,12.0,18.0,New York City,0,...,6.0,6.0,2.0,5.0,3.0,6.0,0.000000,0.000000,0.011494,0.0
366,517373,5.0,3.0,4.0,2.0,25.0,5.0,10.0,Dubai,1,...,6.0,2.0,4.0,3.0,1.0,5.0,0.009259,0.000000,0.011494,0.0
1391,499778,5.0,3.0,4.0,2.0,0.0,1.0,50.0,Paris,0,...,1.0,6.0,3.0,5.0,4.0,2.0,0.000000,0.018692,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756,495084,5.0,4.0,3.0,2.0,31.0,1.0,3.0,New York City,1,...,5.0,2.0,3.0,4.0,1.0,6.0,0.000000,0.000000,0.000000,0.0
930,298063,5.0,4.0,4.0,5.0,2.0,2.0,4.0,New York City,1,...,6.0,2.0,2.0,3.0,2.0,6.0,0.000000,0.000000,0.000000,0.0
1427,499616,5.0,5.0,5.0,4.0,5.0,3.0,10.0,Paris,1,...,4.0,6.0,3.0,5.0,1.0,2.0,0.000000,0.037383,0.000000,0.0
210,212706,5.0,3.0,3.0,2.0,27.0,1.0,2.0,Dubai,1,...,6.0,2.0,3.0,4.0,1.0,5.0,0.009259,0.000000,0.000000,0.0


In [16]:
def standardize(data):
    mean = data.mean()
    std = data.std()
    standardized_data = (data - mean) / std
    return standardized_data

# Standardize each feature column of the train, validation, and test sets
for column in train_data.columns:
    if column != "Label" and column != "id":
        train_data[column] = standardize(train_data[column])

# Standardize each feature column of the validation set
for column in val_data.columns:
    if column != "Label" and column != "id":
        val_data[column] = standardize(val_data[column])

# Standardize each feature column of the test set
for column in test_data.columns:
    if column != "Label" and column != "id":
        test_data[column] = standardize(test_data[column])


In [17]:
# Save the modified dataframe to a new CSV file
train_data.to_csv('training_dataset.csv', index=False)
val_data.to_csv('validation_dataset.csv', index=False)
test_data.to_csv('test_dataset.csv', index=False)

In [18]:
new_df = train_data

label_mapping = {'Dubai': 0, 'Rio de Janeiro': 1, 'New York City': 2, 'Paris': 3}

new_df['Label'] = new_df['Label'].map(label_mapping)

val_data['Label'] = val_data['Label'].map(label_mapping)

test_data['Label'] = test_data['Label'].map(label_mapping)

In [19]:
new_df

,id,Q1,Q2,Q3,Q4,Q7,Q8,Q9,Label,Friends,...,Skyscrapers,Art and Music,Carnival,Cuisine,Sport,Economic,Dubai,Paris,New York City,Rio de Janeiro
290,499778,-0.374437,-1.286625,1.132462,-1.835547,0.516858,0.314218,-0.365891,0,0.619985,...,1.015989,-0.567692,-1.233735,-1.285645,0.407278,0.669608,1.815962,-0.459432,2.329866,-0.5033
131,523461,0.764476,-0.447672,0.250448,-0.281983,0.929076,0.010140,-0.567718,0,0.619985,...,0.540863,-1.856698,-0.059515,0.179930,-0.756373,1.260676,0.614054,-0.459432,-0.580979,-0.5033
778,602496,0.764476,1.230234,1.132462,0.494800,-1.132015,2.442764,0.945981,2,-1.610745,...,1.015989,1.365817,-0.646625,0.912717,-0.174548,1.260676,-0.587854,-0.459432,0.874443,-0.5033
366,517373,0.764476,-0.447672,0.250448,-1.058765,0.929076,0.314218,0.138675,0,0.619985,...,1.015989,-1.212195,0.527595,-0.552857,-1.338199,0.669608,0.614054,-0.459432,0.874443,-0.5033
1391,499778,0.764476,-0.447672,0.250448,-1.058765,-1.132015,-0.902094,4.175204,3,-1.610745,...,-1.359642,1.365817,-0.059515,0.912717,0.407278,-1.103598,-0.587854,2.047805,-0.580979,-0.5033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756,495084,0.764476,0.391281,-0.631565,-1.058765,1.423738,-0.902094,-0.567718,2,0.619985,...,0.540863,-1.212195,-0.059515,0.179930,-1.338199,1.260676,-0.587854,-0.459432,-0.580979,-0.5033
930,298063,0.764476,0.391281,0.250448,1.271582,-0.967127,-0.598016,-0.466805,2,0.619985,...,1.015989,-1.212195,-0.646625,-0.552857,-0.756373,1.260676,-0.587854,-0.459432,-0.580979,-0.5033
1427,499616,0.764476,1.230234,1.132462,0.494800,-0.719796,-0.293938,0.138675,3,0.619985,...,0.065737,1.365817,-0.059515,0.912717,-1.338199,-1.103598,-0.587854,4.555043,-0.580979,-0.5033
210,212706,0.764476,-0.447672,-0.631565,-1.058765,1.093964,-0.902094,-0.668631,0,0.619985,...,1.015989,-1.212195,-0.059515,0.179930,-1.338199,0.669608,0.614054,-0.459432,-0.580979,-0.5033


In [20]:
val_data

,id,Q1,Q2,Q3,Q4,Q7,Q8,Q9,Label,Friends,...,Skyscrapers,Art and Music,Carnival,Cuisine,Sport,Economic,Dubai,Paris,New York City,Rio de Janeiro
686,633746,-1.591477,0.407615,-0.624423,1.223683,-0.478801,-0.618563,-0.352179,1,0.608985,...,-1.315402,-0.678890,1.680774,0.152635,1.017589,-1.147755,-0.615314,-0.550022,-0.560533,1.098016
45,209691,0.768508,1.259238,-0.624423,1.223683,0.466298,0.385886,1.354986,0,-1.639280,...,1.113607,-0.678890,1.088096,0.152635,1.017589,1.274019,0.717488,0.687000,0.839607,1.098016
1326,317266,0.768508,0.407615,0.249471,-0.337614,-1.252064,-0.618563,0.216876,3,-1.639280,...,-0.343798,1.227434,-1.282616,0.861297,0.424671,-1.147755,-0.615314,-0.550022,-0.560533,-0.571765
1261,502015,-2.771469,-1.295633,-1.498317,-1.118262,0.638134,-0.953380,-0.807423,3,0.608985,...,-1.315402,-1.949773,-1.282616,-1.973351,-0.168247,-1.753199,-0.615314,-0.550022,-0.560533,-0.571765
190,522365,0.768508,0.407615,1.123364,-1.118262,-0.049210,-0.283747,-0.579801,0,0.608985,...,1.113607,-1.314332,-0.689938,-1.264689,-0.761165,0.668575,0.717488,0.687000,0.839607,1.098016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1270,411711,0.768508,-1.295633,0.249471,-1.118262,-1.681654,-0.618563,-0.465990,3,-1.639280,...,1.113607,0.591992,-1.282616,0.152635,-0.761165,-0.542312,-0.615314,0.687000,-0.560533,1.098016
1151,398273,0.768508,-1.295633,-0.624423,-0.337614,-0.392883,-0.953380,-0.579801,3,0.608985,...,-0.829600,1.227434,-1.282616,0.861297,0.424671,-0.542312,2.050291,6.872110,2.239746,4.437579
285,601547,-0.411485,-0.444009,-1.498317,1.223683,0.466298,-0.618563,-0.807423,0,0.608985,...,1.113607,-1.314332,-0.097260,-0.556027,-0.761165,1.274019,0.717488,0.687000,0.839607,1.098016
1204,317284,0.768508,1.259238,-0.624423,-1.118262,-0.908391,-0.283747,0.216876,3,-1.639280,...,-0.829600,1.227434,-1.282616,0.861297,-0.168247,0.063132,-0.615314,1.924022,0.839607,1.098016


In [21]:
test_data

,id,Q1,Q2,Q3,Q4,Q7,Q8,Q9,Label,Friends,...,Skyscrapers,Art and Music,Carnival,Cuisine,Sport,Economic,Dubai,Paris,New York City,Rio de Janeiro
989,491031,0.803026,1.239700,0.312864,0.451587,-1.922268,0.631043,-0.096445,2,0.661948,...,1.109090,-0.106974,0.301505,0.149783,0.856390,0.666129,-0.534460,-0.55363,1.257217,-0.563159
227,618161,-0.172551,-1.179361,0.312864,-0.269970,0.425737,0.051272,-0.482227,0,0.661948,...,1.109090,-0.762191,0.301505,-0.497809,-0.228863,1.289829,1.742805,-0.55363,-0.572594,-0.563159
101,520249,-2.123706,-1.985714,-0.554891,-0.991527,-0.116110,0.920928,-0.482227,0,-1.500416,...,1.109090,-1.417407,-0.274096,0.149783,-1.314116,0.666129,1.742805,-0.55363,-0.572594,-0.563159
674,517648,-1.148129,-1.179361,-1.422647,1.173143,1.238508,-0.238613,-0.289336,1,0.661948,...,-1.295574,1.203459,0.877106,0.797376,-0.771490,-1.828671,-0.534460,-0.55363,-0.572594,1.028847
42,492858,0.803026,0.433346,0.312864,1.173143,0.335429,-0.238613,-0.675118,0,0.661948,...,1.109090,-0.762191,-0.849696,1.444969,-1.314116,0.666129,-0.534460,-0.55363,-0.572594,-0.563159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,631752,-1.148129,-1.179361,-1.422647,1.173143,0.335429,-0.238613,-0.289336,1,0.661948,...,-1.295574,0.548242,1.452707,-0.497809,0.313764,-1.204971,-0.534460,-0.55363,-0.572594,2.620853
846,523486,0.803026,0.433346,-0.554891,-0.991527,0.154814,-0.818384,-0.578673,2,0.661948,...,-0.814641,0.548242,0.301505,-0.497809,1.399017,-1.828671,0.604172,-0.55363,3.087027,-0.563159
703,413095,0.803026,0.433346,0.312864,1.173143,0.967584,-0.238613,-0.385782,1,0.661948,...,-0.333709,1.203459,1.452707,-1.145402,1.399017,-1.204971,-0.534460,-0.55363,-0.572594,2.620853
760,525469,-2.123706,-1.179361,-1.422647,-1.713084,-1.109497,0.341158,-0.578673,2,-1.500416,...,0.628157,-0.106974,1.452707,-1.145402,0.313764,-1.204971,-0.534460,-0.55363,-0.572594,-0.563159


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Extracting labels
Y_train = new_df['Label']

# Extracting features
X_train = new_df.drop(['Label', 'id'], axis=1)

features_val = val_data.drop(['Label', 'id'], axis=1).values
labels_val = val_data['Label'].values

# Define different values for the number of neurons in the first Dense layer
neuron_values = [1, 10, 21, 40, 50, 100]

# Initialize variables to store the best validation accuracy and corresponding number of neurons
best_val_accuracy = 0
best_neurons = None
best_weights = None
best_bias = None

# Iterate over different values for the number of neurons
for neurons in neuron_values:
    # Define neural network architecture for classification
    model = Sequential([
        Dense(neurons, activation='relu', input_shape=(21,)),
        Dense(4, activation='softmax')  # Output layer with 4 neurons for classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train the model
    history = model.fit(X_train, Y_train, epochs=200, batch_size=50, validation_data=(features_val, labels_val), verbose=0)

    # Evaluate the model on validation data
    val_loss, val_accuracy = model.evaluate(features_val, labels_val)

    # Check if current model is the best
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_neurons = neurons
        best_weights = model.get_weights()

# Output the best validation accuracy and corresponding number of neurons
print("Best Validation Accuracy:", best_val_accuracy)
print("Corresponding Number of Neurons in the First Dense Layer:", best_neurons)
print("\nWeights and biases at the iteration with the highest validation accuracy:")

for layer_weights in best_weights:
    print(layer_weights)
    print(layer_weights.shape)

19/19 [==============================] - 0s 2ms/step - loss: 0.2921 - accuracy: 0.9489
Best Validation Accuracy: 0.9488926529884338
Corresponding Number of Neurons in the First Dense Layer: 100

Weights and biases at the iteration with the highest validation accuracy:
[[-0.22388998 -0.24073581 -0.08718447 ... -0.0304299   0.2301399
   0.01255585]
 [-0.26259282 -0.14151755  0.02007547 ...  0.54430336  0.38197643
  -0.18776989]
 [ 0.03167151  0.07292522 -0.18592016 ... -0.1787945   0.33439264
   0.13972722]
 ...
 [ 0.22423403  0.1778834   0.3957902  ... -0.00537643 -0.051898
   0.14055695]
 [-0.02066196 -0.4203152   0.01028583 ...  0.3967653   0.29233307
  -0.07454354]
 [ 0.10804898 -0.03199629 -0.14800322 ...  0.22522983  0.19154888
   0.3274995 ]]
(21, 100)
[ 0.06312305 -0.0222272   0.23851313  0.1031492   0.00982504 -0.12555869
  0.19114603 -0.04930691  0.1310916   0.30867872  0.09324113  0.16765791
  0.12632354  0.22794698  0.13425666  0.10131871  0.03084702  0.29078314
  0.13947624 

In [24]:
# Code for Nerual Net with Weights for a two layer neural net using no Tensor flow nor SKlearn

import numpy as np

# Define the ReLU activation function
def relu(x):
    return np.maximum(0, x)

# Define the softmax activation function
def softmax(x):
    exp_vals = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)
# Load the provided weights and biases
W1 = np.array([[-0.18122767, -0.34261945,  0.618089  ,  0.10254677,  0.21396106, -0.20339623,
                 0.291714  , -0.16236162,  0.15367551,  0.54062206, -0.28475922, -0.19027379,
                 0.16373168, -0.09066109, -0.09560236, -0.14449425, -0.05717265,  0.09207235,
                 0.32065564,  0.17228729,  0.9291354 ],
               [-0.06346133, -0.13824923,  0.17418289,  0.12221111, -0.22000366,  0.5430643 ,
                 0.06857064, -0.24972379, -0.24924523,  0.6991673 , -0.02246046, -0.1329794 ,
                 0.2666616 ,  0.43108594, -0.72167367, -0.09078955,  0.05968696, -0.34905773,
                -0.04034884,  0.12455011,  0.5730588 ],
               [-0.01934707, -0.44885972,  0.40595472,  0.6252177 , -0.28318506, -0.23074375,
                 0.08948409,  0.25548714, -0.08429797,  0.27890593, -0.15124547, -0.43240535,
                 0.09891299,  0.4641335 , -0.27890304,  0.01780645, -0.3261511 ,  0.36541602,
                 0.4352592 , -0.5203126 ,  0.3334015 ],
               [-0.10497751,  0.2669729 , -0.21674785, -0.23630731, -0.3270558 , -0.40418202,
                -0.06366117, -0.3645825 ,  0.4404783 ,  0.19042154, -0.1264574 ,  0.11860222,
                 0.0439868 , -0.78384966, -0.01899807,  0.7422485 ,  0.18102828,  0.16726352,
                -0.32497314,  0.06830379, -0.17687565],
               [ 0.4583098 ,  0.5170336 , -0.40581724,  0.65864444, -0.38663328,  0.16330394,
                 0.8627991 , -0.417202  , -0.06232374, -0.46584862, -0.25180084, -0.4554947 ,
                -0.94429016,  0.08913619, -0.22523041,  0.44131273,  0.11915996,  0.23561718,
                 0.24612522,  0.00627808, -0.33742946],
               [ 0.07665887, -0.7970367 , -0.610155  , -0.03483709,  0.19004585,  0.27672246,
                 0.47861266,  0.4090386 ,  0.42219093,  0.05093561,  0.04466511,  0.12740186,
                 0.1669145 , -0.2467548 ,  0.10142637, -0.21916868, -0.7589794 ,  0.36849925,
                 0.14740308, -0.15094109, -0.59036976],
               [-0.26543468,  0.05632016, -0.046246  ,  0.17335907, -0.20859505,  0.23241974,
                 0.44276994, -0.06674672, -0.8130979 , -0.18449211, -0.06467906,  0.06794851,
                -0.03623752,  0.07890465,  0.32165965,  0.05731079,  0.11262164,  0.07633847,
                -0.09712634,  0.30031228, -0.04076015],
               [-0.1039874 ,  0.20159413, -0.3124274 ,  0.37509274, -0.05520559, -0.0891054 ,
                 0.33545583,  0.4608281 ,  0.44231078,  0.3793567 , -0.19503377, -0.3190721 ,
                 0.04294661,  0.42935267, -0.42101023, -0.07647344, -0.1790378 , -0.16546597,
                -0.51683456, -0.13596894,  0.17902759],
               [-0.2388623 , -0.20904703,  0.00366932, -0.17125331,  0.30521524, -0.03484877,
                 0.5331394 ,  0.5780791 , -0.4246371 ,  0.11049909, -0.06383548, -0.05925533,
                 0.1527637 , -0.5447987 , -0.75106645, -0.26508462,  0.3297146 ,  0.19605982,
                -0.31031346,  0.46230417,  0.24369736],
               [-0.21686314,  0.22733536,  0.2802096 ,  0.34134674, -0.03133137, -0.5799788 ,
                 0.09264815,  0.01107777,  0.00419636,  0.19295374,  0.23748627,  0.26386175,
                 0.17136635,  0.16007751, -0.38166624, -0.13047892, -0.01205741,  0.55852216,
                 0.22176605, -0.28423756, -0.04873079],
               [ 0.15703022, -0.023573  , -0.3072977 , -0.13483116, -0.10190468,  0.31820893,
                 0.4010724 ,  0.43832767, -0.06907237,  0.24904378, -0.34712037,  0.18628794,
                -0.07092155,  0.17577222, -0.6740756 ,  0.29538926,  0.06996498,  0.32581928,
                -0.06399068,  0.28447017,  0.23879321],
               [ 0.9832829 ,  0.2911431 , -0.48568362,  0.5333393 ,  0.43005866, -0.00628746,
                 0.19680624, -0.3499347 , -0.9426791 , -0.21479787, -0.23366301,  0.37935752,
                 0.1624602 ,  0.0313803 , -0.22092278, -0.20949374,  0.21032232,  0.03866184,
                -0.13580762,  0.4778398 ,  0.2142836 ],
               [-0.93110865, -0.13042153,  0.6099056 , -0.2766079 , -0.28029045, -0.08692583,
                -0.20239934, -0.18931524,  0.7845982 , -0.45298648, -0.14439292,  0.37843677,
                -0.18610373, -0.04441105, -0.18261507, -0.04567687,  0.5044848 , -0.30623633,
                -0.31990525,  0.38109517,  0.00610555],
               [ 0.19919187, -0.01232368, -0.13178451, -0.11507779, -0.59689564,  0.4055603 ,
                -0.6713709 ,  0.11300497,  0.33723265,  0.18955497,  0.49008515, -0.3684191 ,
                -0.01620064,  0.00996921, -0.30614203,  0.22986032,  0.40966287, -0.13441521,
                 0.17970774, -0.07905452,  0.08160879],
               [ 0.31459197,  0.08974732,  0.6636756 , -0.29172412, -0.17133105, -0.19903652,
                -0.21688837, -0.01175701, -0.0480477 ,  0.20952512, -0.30142784,  0.02113666,
                 0.21085596, -0.12022368, -0.12905148, -0.30907655, -0.10794251, -0.17158492,
                -0.09164512,  0.2618165 , -0.02234494],
               [-0.06111471,  0.45019048, -0.15026124,  0.20540844, -0.8261531 ,  0.47170648,
                 0.0320717 , -0.12562333,  0.40269428,  0.08830403, -0.5466605 , -0.31381488,
                -0.06214912,  0.42273626,  0.05883202,  0.13478488,  0.4299662 ,  0.08323681,
                -0.3379745 , -0.02327799, -0.42244443],
               [ 0.15970357,  0.10833409, -0.38240883, -0.08433139,  0.640397  , -0.43413657,
                 0.26265728, -0.09202445, -0.27852032,  0.06315765, -0.3925917 , -0.01223488,
                 0.00536075,  0.25810766,  0.21727486, -0.36136857, -0.01304609, -0.20058762,
                 0.05342385,  0.22386226,  0.40233874],
               [ 0.9961319 , -0.47614732, -0.04396398, -0.11079986,  0.80423844, -0.44784796,
                 0.7200835 , -0.07598324, -0.12549305, -0.24063627,  0.17702708, -0.46834213,
                -0.16294262, -0.05879875, -0.61824703, -0.31461105, -0.47599941, -0.49110687,
                 0.16232114, -0.5877392 , -0.62978625],
               [-0.21637195, -0.2309664 ,  0.9855049 , -0.4558403 , -0.07418732, -0.485502  ,
                 0.31574628,  0.47733516, -0.07944921, -0.80867535, -0.19387151, -0.10583898,
                -0.16175757, -0.758739  ,  0.42223057, -0.53831255,  0.7728058 ,  0.44382536,
                 0.78479993, -0.18485616, -0.13887733],
               [-0.39282545, -0.128537  , -0.1568943 , -0.8081667 ,  0.344583  ,  0.10867279,
                 0.45028833, -0.37345892, -0.03050019,  0.468901  ,  0.81581634,  0.24736309,
                 0.69012   ,  0.17623691,  0.10395848, -0.28579494, -0.42758644, -0.03471075,
                -0.02715331,  0.5695901 ,  0.4957778 ],
               [-0.15354164,  0.75120705,  0.05050168,  0.159177  , -0.05671156,  0.1182749 ,
                -0.3345349 ,  0.07981967,  0.20751178, -0.23091902,  0.1530818 , -0.41384465,
                -0.10118825, -0.5140071 ,  0.14178313,  0.8507916 , -0.03672163,  0.6105546 ,
                -0.01237321, -0.53478944,  0.1146058 ]])

b1 = np.array([0.324486  , -0.04677757,  0.25435203,  0.54596084, -0.01825691,
               -0.17329064,  0.11099098,  0.0565879 ,  0.08471949,  0.09760601,
               -0.11611599,  0.06128851, -0.09078927,  0.47546253,  0.01965878,
               0.26970643,  0.20456119, -0.11610996,  0.14213483,  0.2837125 ,
               0.42036554])

# Convert the data to a NumPy array
W2 = np.array([[ 1.0872543 , -0.116946  , -0.75965863, -0.0632996 ],
               [-0.29525173,  0.875843  , -0.63909847, -0.41030827],
               [-0.41719016, -1.1689205 , -0.91439945,  1.0372523 ],
               [ 0.37567714,  0.09568357, -0.82579607, -0.3686573 ],
               [ 0.5657957 , -0.42541486,  0.56216604, -1.0856181 ],
               [ 0.01723729,  0.36066487,  0.39403522, -0.93994635],
               [ 0.39989385, -0.8637217 , -0.09325068, -0.12395655],
               [-0.59429395,  0.14000387, -0.3162748 ,  1.1053296 ],
               [-0.0685562 ,  0.20627724, -0.7349946 , -0.6119177 ],
               [ 0.05722361, -0.4267471 ,  0.8884751 , -0.0791458 ],
               [-0.04653866, -0.49968863,  0.87750876, -0.2732686 ],
               [ 0.15538497, -0.13908266,  0.8042803 , -0.06139815],
               [-0.49309307, -0.06041425,  0.48143333, -0.23557398],
               [ 0.55906063, -0.7886456 ,  0.32624957, -0.07232844],
               [-0.563009  ,  0.21733761, -0.03595051,  0.6348426 ],
               [-0.11944032,  0.88420224, -0.00992112, -0.9319595 ],
               [-0.38416424,  0.64416575, -0.60113144,  0.535555  ],
               [-0.61675984,  0.54467523, -0.23977481,  0.6247552 ],
               [ 0.09536663, -0.08351897, -0.7319939 ,  0.8752944 ],
               [-0.39597526, -0.5393011 ,  0.80642897,  0.5016391 ],
               [-0.6412451 , -0.6501586 ,  0.744258  ,  0.11305002]])

b2 = np.array([ 0.20806314, -0.32820094, -0.07501657,  0.22989844])

# Define the neural network using provided weights and biases
def neural_network(x, W1, b1, W2, b2):
    # Input layer
    input_layer = x

    # Hidden layer
    hidden_layer = relu(np.dot(input_layer, W1) + b1)

    # Output layer
    output_layer = softmax(np.dot(hidden_layer, W2) + b2)

    return np.argmax(output_layer, axis=1)

features_test = test_data.drop(['Label', 'id'], axis=1).values
labels_test = test_data['Label'].values

# print(features_test[0])
# print(neural_network(features_test, W1, b1, W2, b2))
# print(labels_test)

correct_predictions = np.sum(neural_network(features_test, W1, b1, W2, b2) == labels_test)
print("The Accuracy on the test data is: " + str(correct_predictions/labels_test.shape[0]))

The Accuracy on the test data is: 0.9455782312925171


DECISION TREE CODE Using SKLEARN

In [25]:
#Qais CODE:
# CUSTOM DECISION TREE CODE:
import numpy as np
import pandas as pd

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

class MyDecisionTreeClassifier:
    def __init__(self, min_samples_split=2, max_depth=100, min_samples_leaf=1, min_impurity_decrease=0.0):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.root = None

    def _gini(self, groups, classes):
        n_instances = float(sum([len(group) for group in groups]))
        gini = 0.0
        for group in groups:
            size = float(len(group))
            if size == 0:
                continue
            score = 0.0
            for class_val in classes:
                p = [row[-1] for row in group].count(class_val) / size
                score += p * p
            gini += (1.0 - score) * (size / n_instances)
        return gini

    def _test_split(self, index, value, dataset):
        left, right = list(), list()
        for row in dataset:
            if row[index] < value:
                left.append(row)
            else:
                right.append(row)
        return left, right

    def _get_split(self, dataset):
        class_values = list(set(row[-1] for row in dataset))
        b_index, b_value, b_score, b_groups = 999, 999, 999, None
        for index in range(len(dataset[0])-1):
            for row in dataset:
                groups = self._test_split(index, row[index], dataset)
                gini = self._gini(groups, class_values)
                if gini < b_score:
                    b_index, b_value, b_score, b_groups = index, row[index], gini, groups
        return {'index':b_index, 'value':b_value, 'groups':b_groups}


    def _to_terminal(self, group):
        outcomes = [row[-1] for row in group]
        return max(set(outcomes), key=outcomes.count)

    def _split(self, node, depth):
        left, right = node['groups']
        del(node['groups'])

        # Check for early stopping conditions
        if depth >= self.max_depth or len(left) < self.min_samples_split or len(right) < self.min_samples_split:
            node['left'] = node['right'] = self._to_terminal(left + right)
            return

        # Process left child
        if len(left) <= self.min_samples_leaf:
            node['left'] = self._to_terminal(left)
        else:
            # Check if split decreases impurity sufficiently
            node['left'] = self._get_split(left)
            self._split(node['left'], depth+1)

        if len(right) <= self.min_samples_leaf:
            node['right'] = self._to_terminal(right)
        else:
            # Check if split decreases impurity sufficiently
            node['right'] = self._get_split(right)
            self._split(node['right'], depth+1)


    def _build_tree(self, train):
        root = self._get_split(train)
        self._split(root, 1)
        return root

    def fit(self, X, y):
        dataset = np.column_stack((X, y))
        self.root = self._build_tree(dataset)

    def _predict(self, node, row):
        if row[node['index']] < node['value']:
            if isinstance(node['left'], dict):
                return self._predict(node['left'], row)
            else:
                return node['left']
        else:
            if isinstance(node['right'], dict):
                return self._predict(node['right'], row)
            else:
                return node['right']

    def predict(self, X):
        predictions = [self._predict(self.root, row) for row in X]
        return np.array(predictions)



In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [27]:
X_train = new_df.drop(['Label', 'id'], axis=1)
Y_train = new_df['Label']

In [28]:
#From LAB 2:

def build_all_models(max_depths,
                     min_samples_split,
                     criterion,
                     X_train,
                     t_train,
                     X_valid,
                     t_valid):
    """
    Parameters:
        `max_depths` - A list of values representing the max_depth values to be
                       try as hyperparameter values
        `min_samples_split` - An list of values representing the min_samples_split
                       values to try as hyperpareameter values
        `criterion` -  A string; either "entropy" or "gini"

    Returns a dictionary, `out`, whose keys are the the hyperparameter choices, and whose values are
    the training and validation accuracies (via the `score()` method).
    In other words, out[(max_depth, min_samples_split)]['val'] = validation score and
                    out[(max_depth, min_samples_split)]['train'] = training score
    For that combination of (max_depth, min_samples_split) hyperparameters.
    """
    out = {}

    for d in max_depths:
        for s in min_samples_split:
            out[(d, s)] = {}
            # Create a DecisionTreeClassifier based on the given hyperparameters and fit it to the data
            tree = DecisionTreeClassifier(criterion=criterion, max_depth=d, min_samples_split=s)
            tree.fit(X_train, t_train)

            # TODO: store the validation and training scores in the `out` dictionary
            out[(d, s)]['val'] = tree.score(X_valid, t_valid)
            out[(d, s)]['train'] = tree.score(X_train, t_train)


    return out

In [29]:
X_val = val_data.drop(['Label', 'id'], axis=1)
Y_val = val_data['Label']

In [30]:
#From LAB 2:
criterions = ["gini"]
max_depths = [1, 5, 10, 15, 20, 25, 30, 50, 100]
min_samples_split = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

for criterion in criterions:
    print("\nUsing criterion {}".format(criterion))
    res = build_all_models(max_depths, min_samples_split, criterion, X_train, Y_train, X_val, Y_val)

    # TODO: complete this loop which should search for the optimal
    best_score = 0
    best_params = None
    # (max_depth, min_samples_split) given this criterion
    for d, s in res:
      train_score = res[(d, s)]['train']
      val_score = res[(d, s)]['val']


      if val_score > best_score:
        best_score = val_score
        best_params = (d, s)

    # Print the best parameters and corresponding scores for this criterion
    if best_params:
        print("Best parameters for criterion {}: max_depth={}, min_samples_split={}".format(criterion, best_params[0], best_params[1]))
        print("Training Score: {:.4f}, Validation Score: {:.4f}".format(res[best_params]['train'], res[best_params]['val']))


Using criterion gini
Best parameters for criterion gini: max_depth=5, min_samples_split=2
Training Score: 0.9060, Validation Score: 0.8688


In [31]:
if best_params:
  custom_tree = MyDecisionTreeClassifier(max_depth=best_params[0], min_samples_split=best_params[1])
else:
  custom_tree = MyDecisionTreeClassifier(max_depth=10, min_samples_split=16)
custom_tree.fit(X_train.values, Y_train.values)

In [32]:
y_train_pred = custom_tree.predict(X_train.values)
accuracy = np.mean(Y_train == y_train_pred)
print(f'Training Accuracy of custom DecisionTree: {accuracy:.2f}')

Training Accuracy of custom DecisionTree: 0.85


In [33]:
X_val = val_data.drop(['Label', 'id'], axis=1)
Y_val = val_data['Label']
X_test = test_data.drop(['Label', 'id'], axis=1)
Y_test = test_data['Label']

In [34]:
# y_val_pred = custom_tree.predict(X_val.values)
# y_test_pred = custom_tree.predict(X_test.values)
y_val_pred = custom_tree.predict(features_val)
y_test_pred = custom_tree.predict(features_test)

In [35]:
accuracy = np.mean(labels_val == y_val_pred)
print(f'Validation Accuracy of custom DecisionTree: {accuracy:.2f}')
accuracy = np.mean(labels_test == y_test_pred)
print(f'Test Accuracy of custom DecisionTree: {accuracy:.2f}')

Validation Accuracy of custom DecisionTree: 0.79
Test Accuracy of custom DecisionTree: 0.84


In [36]:
# Multi Class logistic regression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Define different values of max_iter
max_iter_values = [1, 10, 100, 200, 500, 1000]

# Initialize variables to store best accuracy and corresponding max_iter
best_accuracy = 0
best_max_iter = None
best_weights = None
best_bias = None

# Iterate over different values of max_iter
for max_iter in max_iter_values:
    # Train logistic regression model
    log_reg_model = LogisticRegression(max_iter=max_iter, multi_class='multinomial', solver='lbfgs')
    log_reg_model.fit(X_train, Y_train)

    # Predict on validation set
    labels_val_pred = log_reg_model.predict(features_val)

    # Calculate validation accuracy
    accuracy_valid = accuracy_score(labels_val, labels_val_pred)

    # Check if current model is the best
    if accuracy_valid > best_accuracy:
        best_accuracy = accuracy_valid
        best_max_iter = max_iter
        best_weights = log_reg_model.coef_
        best_bias = log_reg_model.intercept_

# Output the best validation accuracy and corresponding max_iter
print("Best Validation Accuracy:", best_accuracy)
print("Corresponding max_iter:", best_max_iter)
print("\nBest Model Weights:\n", best_weights)
print("\nBest Model Bias:\n", best_bias)

# log_reg_model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs')
# log_reg_model.fit(X_train, Y_train)

# labels_val_pred = log_reg_model.predict(features_val)

# accuracy_valid = accuracy_score(labels_val, labels_val_pred)
# print("Validation Accuracy:", accuracy_valid)

# # Extract weights and bias
# best_weights = log_reg_model.coef_
# best_bias = log_reg_model.intercept_

# print("Best Model Weights:\n", best_weights)
# print("\nBest Model Bias:\n", best_bias)

# # Predictions on test set
# labels_test_pred = log_reg_model.predict(features_test)

# # Calculate accuracy on test set
# accuracy_test = accuracy_score(labels_test, labels_test_pred)
# print("Test Accuracy:", accuracy_test)

Best Validation Accuracy: 0.9454855195911414
Corresponding max_iter: 10

Best Model Weights:
 [[-0.33940309  0.2472193   0.41756706 -0.7656982   1.61477004  0.39096869
  -0.4970602   0.35207605 -0.0062887  -0.16954027 -0.03287046  0.77178306
  -0.38494902 -0.14004905 -0.18282774  0.20539302  0.2995153   1.84622026
  -0.55305452 -0.45486936 -0.69231854]
 [-0.88639385 -0.53627503 -0.4984716   1.04340248  1.1534191  -0.68895761
   0.38715809  0.0628792  -0.31304032  0.10307021 -0.0928541  -0.42923963
  -0.0859707   0.2474699   0.04955615  0.45290179 -0.31838417 -0.66246758
  -0.4210737  -0.56100265  1.38913662]
 [ 0.85147795  0.55685213 -0.51016056  0.00251264 -1.54479917  0.19338522
  -0.18231387  0.0799605   0.38593801 -0.31636708  0.25983365  0.25845924
  -0.05607753 -0.18145167 -0.13861436 -0.36046857  0.39425238 -0.65582316
  -1.15143514  1.47350133 -0.44432552]
 [ 0.37431898 -0.2677964   0.5910651  -0.28021691 -1.22338997  0.1046037
   0.29221597 -0.49491576 -0.06660898  0.38283713 

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/prepro

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 2. Load your dataset
# Example:
# X = Your features
# y = Your labels

# 3. Preprocess data if needed

# List of k values to try
k_values = [1, 5, 10, 20, 40, 50, 100]

best_accuracy = 0
best_k = None

# Iterate through each value of k
for k in k_values:
    # 5. Create KNN classifier
    knn_classifier = KNeighborsClassifier(n_neighbors=k)

    # 6. Train classifier on training set
    knn_classifier.fit(X_train, Y_train)

    # 7. Make predictions on validation set
    y_pred = knn_classifier.predict(features_val)

    # 8. Evaluate performance on validation set
    accuracy = accuracy_score(labels_val, y_pred)
    print(f"Validation Accuracy for k={k}: {accuracy}")

    # Update best accuracy and k if current accuracy is better
    if accuracy != 1 and accuracy > best_accuracy:  # Ensure accuracy is not 100%
        best_accuracy = accuracy
        best_k = k

print(f"\nBest Validation Accuracy: {best_accuracy} with k={best_k}")


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy for k=1: 0.8551959114139693
Validation Accuracy for k=5: 0.8875638841567292


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy for k=10: 0.9011925042589438
Validation Accuracy for k=20: 0.9045996592844975


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy for k=40: 0.9045996592844975
Validation Accuracy for k=50: 0.9063032367972743


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy for k=100: 0.9045996592844975

Best Validation Accuracy: 0.9063032367972743 with k=50


In [38]:
def softmax(Z):
  exp_Z = np.exp(Z)
  return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def predict(X, W, b):
  Z = np.dot(X, W) + b
  Y = softmax(Z)
  return np.argmax(Y, axis=1)

W = np.array([[-0.38424187,  0.27470821,  0.24180391, -0.76164848,  1.35193149,
                0.27501823, -0.2488667 ,  0.07904521, -0.08664699, -0.10449309,
                0.03328805,  1.18127635, -0.20928487, -0.22006895, -0.21312027,
               -0.01205819,  0.64593763,  1.93528628, -0.48720506, -0.57722476,
               -0.38681229],
              [-0.81831389, -0.74516911, -0.23801954,  1.29639795,  0.46220907,
               -0.33120804,  0.09263839,  0.19158507, -0.30632254, -0.06464896,
               -0.08108342, -0.63479272, -0.31964577,  0.77065523, -0.15201112,
                0.54521492, -0.57524453, -0.98793121, -0.26146121, -0.77358644,
                1.45027956],
              [ 0.46156605,  0.80787978, -0.62665195, -0.31359498, -1.06805551,
                0.43446575,  0.10084344,  0.06398788,  0.42949527, -0.23675026,
                0.1365574 ,  0.09226225,  0.12548032, -0.42035009, -0.18451902,
               -0.39317318,  0.38821869, -0.44532494, -1.30661694,  1.49436843,
               -0.51903302],
              [ 0.74098971, -0.33741888,  0.62286758, -0.22115449, -0.74608505,
               -0.37827594,  0.05538487, -0.33461816, -0.03652574,  0.40589232,
               -0.08876203, -0.63874588,  0.40345032, -0.13023619,  0.54965041,
               -0.13998354, -0.45891178, -0.50203013,  2.05528321, -0.14355722,
               -0.54443426]])

b = np.array([0.4515672, -1.18493688, 0.3048758, 0.42849387])

output_train = predict(X_train, W.T, b)

correct_predictions_train = np.sum(output_train == Y_train)
print("The Accuracy on the train data is: " + str(correct_predictions_train/Y_train.shape[0]))

output_val = predict(features_val, W.T, b)

correct_predictions_val = np.sum(output_val == labels_val)
print("The Accuracy on the validation data is: " + str(correct_predictions_val/labels_val.shape[0]))

output_test = predict(features_test, W.T, b)

correct_predictions_test = np.sum(output_test == labels_test)
print("The Accuracy on the test data is: " + str(correct_predictions_test/labels_test.shape[0]))

The Accuracy on the train data is: 0.9196185286103542
The Accuracy on the validation data is: 0.9522998296422487
The Accuracy on the test data is: 0.9319727891156463
